# DC clean sub-08 released-embedding diffusion prior training

这版不使用自己的 `40.pth`，只使用作者发布的 EEG embeddings 训练新的 diffusion prior。默认只训练并保存；推理 block 保留为注释。

## 1. 路径和运行参数

In [ ]:
import os
import sys

# 国内服务器建议保留；模型已缓存时会直接走缓存。
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
os.environ.setdefault("HF_HOME", "/data/gaoy/projects/.cache/huggingface")
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/data/gaoy/projects/.cache/huggingface/hub")
os.environ.setdefault("TORCH_HOME", "/data/gaoy/.cache/torch")

REPO_ROOT = "/data/gaoy/projects/EEG_Image_decode"
GENERATION_DIR = "/data/gaoy/projects/EEG_Image_decode/Generation"
DATA_ROOT = "/data/gaoy/projects/datasets/EEG_Image_decode"

SUBJECT = "sub-08"
DEVICE = "cuda:0"
SEED = 42

TRAIN_IMAGE_DIR = f"{DATA_ROOT}/images_set/training_images"
TEST_IMAGE_DIR = f"{DATA_ROOT}/images_set/test_images"
VIT_TRAIN_FEATURES = f"{DATA_ROOT}/ViT-H-14_features_train.pt"
VIT_TEST_FEATURES = f"{DATA_ROOT}/ViT-H-14_features_test.pt"

# 作者发布的 EEG embeddings；这版训练只用它，不用 40.pth。
RELEASED_EEG_TRAIN = f"{DATA_ROOT}/emb_eeg/ATM_S_eeg_features_sub-08.pt"
RELEASED_EEG_TEST = f"{DATA_ROOT}/emb_eeg/ATM_S_eeg_features_sub-08_test.pt"

# 新训练出来的 diffusion prior 保存到这里，避免覆盖作者 released 权重。
DIFFUSION_PRIOR_CKPT = f"{DATA_ROOT}/fintune_ckpts/sub-08/diffusion_prior_released_eeg_150epochs.pt"

# 推理输出目录；默认推理 block 是注释状态。
OUTPUT_DIR = f"{DATA_ROOT}/generated_imgs/sub-08_released_eeg_new_prior"

NUM_EPOCHS = 150
LEARNING_RATE = 1e-3
TRAIN_BATCH_SIZE = 1024
NUM_WORKERS = 8
START_INDEX = 0
NUM_CONCEPTS = 200
REPEATS_PER_CONCEPT = 10

for p in [REPO_ROOT, GENERATION_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

for p in [TRAIN_IMAGE_DIR, TEST_IMAGE_DIR, VIT_TRAIN_FEATURES, VIT_TEST_FEATURES, RELEASED_EEG_TRAIN, RELEASED_EEG_TEST]:
    assert os.path.exists(p), p

print("save diffusion prior to:", DIFFUSION_PRIOR_CKPT)
print("optional output dir:", OUTPUT_DIR)

## 2. 导入库并固定随机种子

In [ ]:
import random
import numpy as np
import torch
from torch.utils.data import DataLoader

from diffusion_prior import EmbeddingDataset, DiffusionPriorUNet, Pipe

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")
print("device:", device)
print("cuda count:", torch.cuda.device_count())

## 3. 读取作者 EEG embeddings 和图像 CLIP features

In [ ]:
def load_torch_features(path, key=None):
    obj = torch.load(path, map_location="cpu")
    if key is not None and isinstance(obj, dict):
        return obj[key]
    return obj

def image_files(folder):
    return sorted([
        f for f in os.listdir(folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ])

def collect_image_paths(root):
    folders = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
    folders.sort()
    names = [folder[folder.index("_") + 1:] if "_" in folder else folder for folder in folders]
    paths = []
    for folder in folders:
        folder_path = os.path.join(root, folder)
        for image_name in image_files(folder_path):
            paths.append(os.path.join(folder_path, image_name))
    return folders, names, paths

train_folders, train_concepts, train_image_paths = collect_image_paths(TRAIN_IMAGE_DIR)
test_folders, test_concepts, test_image_paths = collect_image_paths(TEST_IMAGE_DIR)

emb_img_train = load_torch_features(VIT_TRAIN_FEATURES, "img_features").float()
emb_img_test = load_torch_features(VIT_TEST_FEATURES, "img_features").float()
eeg_features_train = load_torch_features(RELEASED_EEG_TRAIN).float()
eeg_features_test = load_torch_features(RELEASED_EEG_TEST).float()

assert len(train_folders) == 1654, len(train_folders)
assert len(test_folders) == 200, len(test_folders)
assert len(train_image_paths) == 1654 * 10, len(train_image_paths)
assert len(test_image_paths) == 200, len(test_image_paths)
assert list(emb_img_train.shape) == [16540, 1024], emb_img_train.shape
assert list(emb_img_test.shape) == [200, 1024], emb_img_test.shape
assert list(eeg_features_train.shape) == [66160, 1024], eeg_features_train.shape
assert list(eeg_features_test.shape) == [200, 1024], eeg_features_test.shape

print("train concepts/images:", len(train_folders), len(train_image_paths))
print("test concepts/images:", len(test_folders), len(test_image_paths))
print("emb_img_train:", emb_img_train.shape)
print("emb_img_test:", emb_img_test.shape)
print("released eeg train:", eeg_features_train.shape)
print("released eeg test:", eeg_features_test.shape)

## 4. 检查训练配对

In [ ]:
emb_img_train_by_author_repeat = emb_img_train.view(1654, 10, 1, 1024).repeat(1, 1, 4, 1).view(-1, 1024)
emb_img_train_4 = emb_img_train.view(1654, 10, 1024).repeat_interleave(4, dim=1).view(-1, 1024)
assert torch.allclose(emb_img_train_4, emb_img_train_by_author_repeat), "repeat_interleave does not match author repeat."
assert eeg_features_train.shape[0] == emb_img_train_4.shape[0]

def print_pairing_debug(n=20):
    print("RELEASED TRAIN EEG embedding -> image/CLIP pairing")
    for sample_idx in range(n):
        image_idx = sample_idx // 4
        rep_idx = sample_idx % 4
        class_idx = image_idx // 10
        image_in_class = image_idx % 10
        assert torch.allclose(emb_img_train_4[sample_idx], emb_img_train[image_idx]), sample_idx
        print(
            f"sample={sample_idx:03d} class={class_idx:04d} image_idx={image_idx:04d} "
            f"image_in_class={image_in_class} rep={rep_idx} path={train_image_paths[image_idx]}"
        )

    print("\nRELEASED TEST EEG embedding -> concept / ground-truth pairing")
    for sample_idx in range(n):
        print(
            f"sample={sample_idx:03d} concept={test_concepts[sample_idx]} "
            f"gt={test_image_paths[sample_idx]} generated={os.path.join(OUTPUT_DIR, test_concepts[sample_idx], '0.png')}"
        )

print_pairing_debug(20)

## 5. 训练并保存新的 diffusion prior

In [ ]:
diffusion_prior = DiffusionPriorUNet(cond_dim=1024, dropout=0.1)
print("diffusion prior parameters:", sum(p.numel() for p in diffusion_prior.parameters() if p.requires_grad))
pipe = Pipe(diffusion_prior, device=device)

train_dataset = EmbeddingDataset(c_embeddings=eeg_features_train, h_embeddings=emb_img_train_4)
train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

pipe.train(train_loader, num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE)

os.makedirs(os.path.dirname(DIFFUSION_PRIOR_CKPT), exist_ok=True)
torch.save(pipe.diffusion_prior.state_dict(), DIFFUSION_PRIOR_CKPT)
print("saved diffusion prior:", DIFFUSION_PRIOR_CKPT)
print("exists:", os.path.exists(DIFFUSION_PRIOR_CKPT))
print("size MB:", os.path.getsize(DIFFUSION_PRIOR_CKPT) / 1024 / 1024)

## 6. 可选：推理生成图片（默认整块注释）

In [ ]:
# 默认不推理。需要用新 prior 生成图片时，取消本 block 的注释。
# from custom_pipeline import Generator4Embeds
#
# assert os.path.exists(DIFFUSION_PRIOR_CKPT), DIFFUSION_PRIOR_CKPT
# pipe.diffusion_prior.load_state_dict(torch.load(DIFFUSION_PRIOR_CKPT, map_location=device))
# pipe.diffusion_prior.eval()
# print("loaded diffusion prior:", DIFFUSION_PRIOR_CKPT)
#
# os.makedirs(OUTPUT_DIR, exist_ok=True)
# generator = Generator4Embeds(num_inference_steps=4, device=device)
#
# end_index = min(START_INDEX + NUM_CONCEPTS, len(test_concepts), eeg_features_test.shape[0])
# for k in range(START_INDEX, end_index):
#     prior_generator = torch.Generator(device=device).manual_seed(SEED + k)
#     eeg_embeds = eeg_features_test[k:k + 1].to(device)
#     h = pipe.generate(c_embeds=eeg_embeds, num_inference_steps=50, guidance_scale=5.0, generator=prior_generator)
#
#     for j in range(REPEATS_PER_CONCEPT):
#         image_generator = torch.Generator(device=device).manual_seed(SEED + k * 1000 + j)
#         image = generator.generate(h.to(dtype=torch.float16), generator=image_generator)
#         path = os.path.join(OUTPUT_DIR, test_concepts[k], f"{j}.png")
#         os.makedirs(os.path.dirname(path), exist_ok=True)
#         image.save(path)
#         print("Image saved to", path)